# CUDA Tutorial: Writing GPU Kernels in C/C++

## What is CUDA?

**CUDA** is NVIDIA's parallel computing platform. It lets you write GPU kernels in C/C++ with full control over threads, memory, and synchronization.

### Why learn CUDA?

| | PyTorch | Triton | CUDA |
|---|---|---|---|
| **Ease** | Easiest | Medium | Hardest |
| **Control** | Least | Good | Full |
| **Performance** | Good | Great | Best |
| **Use case** | Standard ops | Custom fused ops | Maximum perf |

### Key Concept: Thread Hierarchy

Unlike Triton where you think in **blocks of data**, CUDA operates on individual **threads**:

```
Grid (all threads for one kernel launch)
├── Block (0,0)          ← threads in a block can cooperate via shared memory
│   ├── Thread 0
│   ├── Thread 1
│   ├── ...
│   └── Thread 255
├── Block (1,0)
│   ├── Thread 0
│   └── ...
└── Block (N,0)
    └── ...
```

- **Thread** → processes one (or a few) elements
- **Block** → group of threads (up to 1024) that share fast memory and can synchronize
- **Grid** → all blocks for a kernel launch
- **Warp** → 32 threads that execute in lockstep (the real hardware unit)

### CUDA vs Triton — Mental Model

| CUDA | Triton |
|---|---|
| `threadIdx.x` — which thread am I in my block? | (handled by compiler) |
| `blockIdx.x` — which block am I? | `tl.program_id(0)` |
| `blockDim.x` — threads per block | `BLOCK_SIZE` (constexpr) |
| Manual shared memory (`__shared__`) | Automatic SRAM usage |
| Manual `__syncthreads()` | Automatic synchronization |
| You manage threads | You manage data tiles |

In [ ]:
import torch
from torch.utils.cpp_extension import load_inline
import time
import os

os.environ.setdefault('CUDA_HOME', '/usr/local/cuda')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

---
## How We Run CUDA in This Notebook

We use `torch.utils.cpp_extension.load_inline` to compile CUDA C++ code at runtime and call it from Python. This gives us:
- Real CUDA kernels (compiled by `nvcc`)
- Easy PyTorch tensor interop
- No need for Makefiles or build systems

The pattern is:
```python
cuda_src = '''
__global__ void my_kernel(float* x, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) x[idx] *= 2;
}

torch::Tensor my_function(torch::Tensor x) {
    // launch kernel, return result
}
'''
module = load_inline(name='my_module', cpp_sources='', cuda_sources=cuda_src, ...)
result = module.my_function(input_tensor)
```

---
## 1. Vector Addition — Your First CUDA Kernel

The simplest kernel: `output[i] = x[i] + y[i]`

### Anatomy of a CUDA Kernel

```cpp
__global__ void add_kernel(float* x, float* y, float* out, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;  // global thread index
    if (idx < n)                                       // bounds check
        out[idx] = x[idx] + y[idx];                    // one thread = one element
}
```

Key elements:
1. **`__global__`** — marks a function as a GPU kernel (callable from CPU)
2. **`blockIdx.x`** — which block this thread belongs to (like `tl.program_id` in Triton)
3. **`threadIdx.x`** — which thread within the block
4. **`blockDim.x`** — number of threads per block
5. **`idx < n`** — boundary check (like `mask` in Triton)

### How global thread index works

```
blockDim.x = 256 (threads per block)

Block 0:  threads [0..255]    →  idx = 0*256 + threadIdx
Block 1:  threads [0..255]    →  idx = 1*256 + threadIdx
Block 2:  threads [0..255]    →  idx = 2*256 + threadIdx
...
```

In [ ]:
add_cuda_src = r'''
#include <torch/extension.h>

// GPU kernel: each thread adds one element
__global__ void add_kernel(const float* x, const float* y, float* output, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        output[idx] = x[idx] + y[idx];
    }
}

// C++ wrapper callable from Python
torch::Tensor add_cuda(torch::Tensor x, torch::Tensor y) {
    TORCH_CHECK(x.is_cuda() && y.is_cuda(), "Inputs must be CUDA tensors");
    auto output = torch::empty_like(x);
    int n = x.numel();

    // Launch configuration: how many threads and blocks?
    int threads_per_block = 256;
    int num_blocks = (n + threads_per_block - 1) / threads_per_block;  // ceiling division

    // Launch!  <<<num_blocks, threads_per_block>>>
    add_kernel<<<num_blocks, threads_per_block>>>(
        x.data_ptr<float>(), y.data_ptr<float>(),
        output.data_ptr<float>(), n
    );

    return output;
}
'''

add_module = load_inline(
    name='add_module',
    cpp_sources='torch::Tensor add_cuda(torch::Tensor x, torch::Tensor y);',
    cuda_sources=add_cuda_src,
    functions=['add_cuda'],
    verbose=False,
)

# Test
torch.manual_seed(0)
size = 98_432  # intentionally not a multiple of 256
x = torch.rand(size, device='cuda')
y = torch.rand(size, device='cuda')

cuda_output = add_module.add_cuda(x, y)
torch_output = x + y

print(f"Input size: {size} (not a multiple of 256 — tests boundary check)")
print(f"Threads per block: 256")
print(f"Number of blocks: {(size + 255) // 256}")
print(f"Total threads launched: {((size + 255) // 256) * 256}")
print(f"Wasted threads: {((size + 255) // 256) * 256 - size}")
print(f"Max difference: {(cuda_output - torch_output).abs().max():.2e}")
print("Correct!" if torch.allclose(cuda_output, torch_output) else "MISMATCH!")

### Comparison with Triton

| Aspect | CUDA | Triton |
|---|---|---|
| Granularity | 1 thread = 1 element | 1 program = 1024 elements |
| Bounds check | `if (idx < n)` | `mask = offsets < n` |
| Launch config | `<<<blocks, threads>>>` | `kernel[grid](...)` |
| Grid calc | `(n + 255) / 256` | `triton.cdiv(n, BLOCK_SIZE)` |
| Language | C/C++ | Python |

---
## 2. GPU Memory Hierarchy — The Key to Performance

Understanding memory is **the most important thing** in CUDA programming.

```
┌──────────────────────────────────────────────────┐
│                   GPU Chip                       │
│                                                  │
│  ┌──────── SM (Streaming Multiprocessor) ──────┐ │
│  │                                             │ │
│  │  ┌─────────┐  ┌─────────┐                  │ │
│  │  │Registers│  │Registers│   ← Fastest      │ │
│  │  │(per thd)│  │(per thd)│     ~1 cycle     │ │
│  │  └─────────┘  └─────────┘                  │ │
│  │                                             │ │
│  │  ┌───────────────────────┐                  │ │
│  │  │    Shared Memory      │   ← Fast        │ │
│  │  │  (per block, ~100KB)  │     ~5 cycles   │ │
│  │  └───────────────────────┘                  │ │
│  │                                             │ │
│  │  ┌───────────────────────┐                  │ │
│  │  │     L1 Cache          │   ← ~30 cycles  │ │
│  │  └───────────────────────┘                  │ │
│  └─────────────────────────────────────────────┘ │
│                                                  │
│  ┌───────────────────────────────────────────┐   │
│  │              L2 Cache                     │   │
│  └───────────────────────────────────────────┘   │
│                                                  │
│  ┌───────────────────────────────────────────┐   │
│  │          Global Memory (VRAM)             │   │  ← Slowest
│  │          (e.g. 24 GB on RTX 5090)         │   │     ~400 cycles
│  └───────────────────────────────────────────┘   │
└──────────────────────────────────────────────────┘
```

### Memory types in CUDA code

| Type | Declaration | Scope | Speed | Size |
|---|---|---|---|---|
| Register | `int x = 5;` | Per thread | Fastest | ~255 per thread |
| Shared | `__shared__ float s[256];` | Per block | Fast | ~48-100 KB |
| Global | Kernel parameters (pointers) | All threads | Slow | GB |
| Constant | `__constant__ float c[64];` | All threads | Fast (cached) | 64 KB |

---
## 3. Shared Memory — Cooperative Threads

Shared memory is the CUDA feature that Triton hides from you. It's fast on-chip memory
shared by all threads in a block. Use it when:
- Multiple threads need the same data (avoid redundant global loads)
- You need to communicate between threads in a block

### Example: Vector Reduction (Sum)

Summing an array is surprisingly hard to parallelize. The classic approach uses
shared memory with a tree-reduction pattern:

```
Step 0:  [a0, a1, a2, a3, a4, a5, a6, a7]   (8 elements, 4 active threads)
Step 1:  [a0+a1, _, a2+a3, _, a4+a5, _, a6+a7, _]   (2 active threads)
Step 2:  [a0+a1+a2+a3, _, _, _, a4+a5+a6+a7, _, _, _]
Step 3:  [a0+a1+a2+a3+a4+a5+a6+a7, ...]   ← final result
```

In [ ]:
reduce_cuda_src = r'''
#include <torch/extension.h>

// Each block reduces its chunk of data to a single value using shared memory.
// Then we sum the per-block results on the CPU (or with a second kernel).
__global__ void sum_reduce_kernel(const float* input, float* partial_sums, int n) {
    extern __shared__ float sdata[];  // dynamically-sized shared memory

    int tid = threadIdx.x;
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // Load from global to shared memory (0 if out of bounds)
    sdata[tid] = (idx < n) ? input[idx] : 0.0f;
    __syncthreads();  // all threads must finish loading before we reduce

    // Tree reduction in shared memory
    for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
        if (tid < stride) {
            sdata[tid] += sdata[tid + stride];
        }
        __syncthreads();  // wait for all threads before next level
    }

    // Thread 0 of each block writes the block's partial sum
    if (tid == 0) {
        partial_sums[blockIdx.x] = sdata[0];
    }
}

torch::Tensor sum_reduce(torch::Tensor input) {
    TORCH_CHECK(input.is_cuda(), "Input must be CUDA");
    int n = input.numel();
    int threads = 256;
    int blocks = (n + threads - 1) / threads;

    auto partial = torch::zeros(blocks, input.options());

    // shared memory size = threads * sizeof(float)
    sum_reduce_kernel<<<blocks, threads, threads * sizeof(float)>>>(
        input.data_ptr<float>(), partial.data_ptr<float>(), n
    );

    // Sum the partial results (small enough for CPU or a second kernel)
    return partial.sum().unsqueeze(0);
}
'''

reduce_module = load_inline(
    name='reduce_module',
    cpp_sources='torch::Tensor sum_reduce(torch::Tensor input);',
    cuda_sources=reduce_cuda_src,
    functions=['sum_reduce'],
    verbose=False,
)

x = torch.rand(100_000, device='cuda')
cuda_sum = reduce_module.sum_reduce(x)
torch_sum = x.sum()

print(f"CUDA sum:    {cuda_sum.item():.4f}")
print(f"PyTorch sum: {torch_sum.item():.4f}")
print(f"Difference:  {abs(cuda_sum.item() - torch_sum.item()):.2e}")
print("Correct!" if abs(cuda_sum.item() - torch_sum.item()) < 0.1 else "MISMATCH!")
print()
print("Key concepts demonstrated:")
print("  - __shared__ memory for fast inter-thread communication")
print("  - __syncthreads() to ensure all threads are in sync")
print("  - Tree reduction pattern (log2 steps)")
print("  - extern __shared__ for dynamic shared memory allocation")

---
## 4. Fused Softmax — Kernel Fusion in CUDA

Just like in the Triton tutorial, we'll fuse `max → subtract → exp → sum → divide` into one kernel.
In CUDA, we need to manage shared memory and `__syncthreads()` explicitly.

Strategy: **one block per row**, threads cooperate within the block.

```
Row [a, b, c, d, e, f, g, h]      (8 columns, 8 threads in a block)
  ├─ Step 1: Parallel max reduction in shared memory → row_max
  ├─ Step 2: Each thread computes exp(x[i] - row_max)
  ├─ Step 3: Parallel sum reduction in shared memory → sum_exp
  └─ Step 4: Each thread writes exp_val / sum_exp
```

In [ ]:
softmax_cuda_src = r'''
#include <torch/extension.h>
#include <float.h>
#include <math.h>

// One block per row. Each thread handles one (or more) columns.
// Uses shared memory for reductions (max, sum).
__global__ void softmax_kernel(
    const float* input, float* output,
    int n_cols
) {
    extern __shared__ float sdata[];

    int row = blockIdx.x;
    int tid = threadIdx.x;
    int row_offset = row * n_cols;

    // --- Step 1: Find row max (for numerical stability) ---
    float thread_max = -FLT_MAX;
    for (int col = tid; col < n_cols; col += blockDim.x) {
        float val = input[row_offset + col];
        thread_max = fmaxf(thread_max, val);
    }
    sdata[tid] = thread_max;
    __syncthreads();

    // Parallel max reduction
    for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
        if (tid < stride)
            sdata[tid] = fmaxf(sdata[tid], sdata[tid + stride]);
        __syncthreads();
    }
    float row_max = sdata[0];
    __syncthreads();

    // --- Step 2: Compute exp(x - max) and partial sum ---
    float thread_sum = 0.0f;
    for (int col = tid; col < n_cols; col += blockDim.x) {
        float val = expf(input[row_offset + col] - row_max);
        output[row_offset + col] = val;  // temporarily store exp values
        thread_sum += val;
    }
    sdata[tid] = thread_sum;
    __syncthreads();

    // Parallel sum reduction
    for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
        if (tid < stride)
            sdata[tid] += sdata[tid + stride];
        __syncthreads();
    }
    float sum_exp = sdata[0];
    __syncthreads();

    // --- Step 3: Normalize ---
    for (int col = tid; col < n_cols; col += blockDim.x) {
        output[row_offset + col] /= sum_exp;
    }
}

torch::Tensor softmax_cuda(torch::Tensor input) {
    TORCH_CHECK(input.dim() == 2 && input.is_cuda(), "Need 2D CUDA tensor");
    int n_rows = input.size(0);
    int n_cols = input.size(1);
    auto output = torch::empty_like(input);

    int threads = std::min(1024, n_cols);
    // Round up to nearest power of 2 for clean reductions
    threads = 1;
    while (threads < n_cols && threads < 1024) threads <<= 1;

    // One block per row, shared memory for reductions
    softmax_kernel<<<n_rows, threads, threads * sizeof(float)>>>(
        input.data_ptr<float>(), output.data_ptr<float>(), n_cols
    );
    return output;
}
'''

softmax_module = load_inline(
    name='softmax_module',
    cpp_sources='torch::Tensor softmax_cuda(torch::Tensor input);',
    cuda_sources=softmax_cuda_src,
    functions=['softmax_cuda'],
    verbose=False,
)

x = torch.randn(128, 512, device='cuda')
cuda_out = softmax_module.softmax_cuda(x)
torch_out = torch.softmax(x, dim=1)

print(f"Input shape: {x.shape}")
print(f"Max diff: {(cuda_out - torch_out).abs().max():.2e}")
print(f"Rows sum to 1? {cuda_out.sum(dim=1)[:5].tolist()}")
print("Correct!" if torch.allclose(cuda_out, torch_out, atol=1e-5) else "MISMATCH!")
print()
print("Notice how much more code CUDA needs vs Triton for the same operation!")
print("CUDA: explicit shared memory, __syncthreads, manual reductions")
print("Triton: just tl.max(), tl.sum() — compiler handles the rest")

---
## 5. Matrix Multiplication — Tiled with Shared Memory

The canonical GPU algorithm. Each block computes a `TILE_SIZE × TILE_SIZE` output tile.
Tiles of A and B are loaded into shared memory to reduce global memory traffic.

### The Tiling Strategy

```
Without tiling: each thread loads its own data from slow global memory
                → N² memory reads for N² output elements (bad reuse)

With tiling:    threads cooperatively load tiles into shared memory
                → each global load is reused by TILE_SIZE threads

    A (M×K)              B (K×N)              C (M×N)
    ┌──┬──┬──┐          ┌──┬──┬──┐          ┌──┬──┬──┐
    │  │██│  │          │  │  │  │          │  │  │  │
    ├──┼──┼──┤    ×     │──│──│──│    =     │──│──│──│
    │  │██│  │          │██│██│██│          │  │██│  │
    ├──┼──┼──┤          ├──┼──┼──┤          ├──┼──┼──┤
    │  │██│  │          │  │  │  │          │  │  │  │
    └──┴──┴──┘          └──┴──┴──┘          └──┴──┴──┘
       ██ = tiles loaded into shared memory for one step
       Block computes the ██ tile of C by iterating over K
```

In [ ]:
matmul_cuda_src = r'''
#include <torch/extension.h>

#define TILE_SIZE 32

// Tiled matrix multiplication: C = A × B
// Each block computes a TILE_SIZE × TILE_SIZE tile of C.
// Each thread computes one element of C.
__global__ void matmul_kernel(
    const float* A, const float* B, float* C,
    int M, int N, int K
) {
    // Shared memory tiles for A and B
    __shared__ float As[TILE_SIZE][TILE_SIZE];
    __shared__ float Bs[TILE_SIZE][TILE_SIZE];

    int row = blockIdx.y * TILE_SIZE + threadIdx.y;  // output row
    int col = blockIdx.x * TILE_SIZE + threadIdx.x;  // output col

    float acc = 0.0f;

    // Loop over tiles along K dimension
    for (int t = 0; t < (K + TILE_SIZE - 1) / TILE_SIZE; t++) {
        // Cooperatively load tiles into shared memory
        int a_col = t * TILE_SIZE + threadIdx.x;
        int b_row = t * TILE_SIZE + threadIdx.y;

        As[threadIdx.y][threadIdx.x] = (row < M && a_col < K)
            ? A[row * K + a_col] : 0.0f;
        Bs[threadIdx.y][threadIdx.x] = (b_row < K && col < N)
            ? B[b_row * N + col] : 0.0f;

        __syncthreads();  // wait for all loads

        // Multiply tile: each thread accumulates its dot product
        for (int k = 0; k < TILE_SIZE; k++) {
            acc += As[threadIdx.y][k] * Bs[k][threadIdx.x];
        }

        __syncthreads();  // wait before loading next tile
    }

    // Write result
    if (row < M && col < N) {
        C[row * N + col] = acc;
    }
}

torch::Tensor matmul_cuda(torch::Tensor a, torch::Tensor b) {
    TORCH_CHECK(a.size(1) == b.size(0), "Incompatible dimensions");
    int M = a.size(0), K = a.size(1), N = b.size(1);
    auto c = torch::zeros({M, N}, a.options());

    // 2D grid of 2D blocks
    dim3 threads(TILE_SIZE, TILE_SIZE);  // 32×32 = 1024 threads per block
    dim3 blocks(
        (N + TILE_SIZE - 1) / TILE_SIZE,
        (M + TILE_SIZE - 1) / TILE_SIZE
    );

    matmul_kernel<<<blocks, threads>>>(
        a.data_ptr<float>(), b.data_ptr<float>(),
        c.data_ptr<float>(), M, N, K
    );
    return c;
}
'''

matmul_module = load_inline(
    name='matmul_module',
    cpp_sources='torch::Tensor matmul_cuda(torch::Tensor a, torch::Tensor b);',
    cuda_sources=matmul_cuda_src,
    functions=['matmul_cuda'],
    verbose=False,
)

M, N, K = 512, 256, 128
a = torch.randn(M, K, device='cuda')
b = torch.randn(K, N, device='cuda')

cuda_out = matmul_module.matmul_cuda(a, b)
torch_out = a @ b

print(f"Shape: ({M},{K}) @ ({K},{N}) -> ({M},{N})")
print(f"Tile size: {32}×{32}, Grid: {(N+31)//32}×{(M+31)//32} blocks")
print(f"Each block: 1024 threads cooperating via shared memory")
print(f"Max diff: {(cuda_out - torch_out).abs().max():.2e}")
print("Correct!" if torch.allclose(cuda_out, torch_out, atol=1e-2) else "MISMATCH!")

### Why Tiling Helps — Memory Access Math

Without tiling (naive): each thread loads one row of A and one column of B
- Total global loads = M × N × K × 2 (read A[i,k] and B[k,j] for each k)

With tiling (TILE_SIZE = T): tiles are loaded into shared memory and reused
- Total global loads = M × N × K × 2 / T  (each load is reused T times)
- For T=32, that's a **32× reduction** in global memory traffic!

---
## 6. Synchronization & Atomics

### `__syncthreads()`
- Barrier for all threads **within a block**
- All threads must reach it before any can proceed
- **Cannot** synchronize across blocks (blocks are independent)

### Atomic Operations
- When multiple threads write to the same location, use atomics
- `atomicAdd(&addr, val)` — guaranteed thread-safe addition
- Slower than regular operations, use only when necessary

### Example: Histogram with Atomics

In [ ]:
histogram_cuda_src = r'''
#include <torch/extension.h>

// Compute histogram of integer values in [0, num_bins)
// Multiple threads may increment the same bin → need atomicAdd
__global__ void histogram_kernel(
    const int* data, int* bins,
    int n, int num_bins
) {
    // Use shared memory for per-block histogram (much faster than global atomics)
    extern __shared__ int local_bins[];

    // Initialize local bins to 0
    for (int i = threadIdx.x; i < num_bins; i += blockDim.x)
        local_bins[i] = 0;
    __syncthreads();

    // Count into shared memory (fast atomics)
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        int bin = data[idx];
        if (bin >= 0 && bin < num_bins)
            atomicAdd(&local_bins[bin], 1);
    }
    __syncthreads();

    // Merge local bins into global bins (fewer global atomics)
    for (int i = threadIdx.x; i < num_bins; i += blockDim.x) {
        if (local_bins[i] > 0)
            atomicAdd(&bins[i], local_bins[i]);
    }
}

torch::Tensor histogram_cuda(torch::Tensor data, int num_bins) {
    TORCH_CHECK(data.is_cuda() && data.dtype() == torch::kInt32, "Need CUDA int32");
    int n = data.numel();
    auto bins = torch::zeros(num_bins, data.options());

    int threads = 256;
    int blocks = (n + threads - 1) / threads;

    histogram_kernel<<<blocks, threads, num_bins * sizeof(int)>>>(
        data.data_ptr<int>(), bins.data_ptr<int>(), n, num_bins
    );
    return bins;
}
'''

histogram_module = load_inline(
    name='histogram_module',
    cpp_sources='torch::Tensor histogram_cuda(torch::Tensor data, int num_bins);',
    cuda_sources=histogram_cuda_src,
    functions=['histogram_cuda'],
    verbose=False,
)

# Generate random integers in [0, 10)
data = torch.randint(0, 10, (1_000_000,), device='cuda', dtype=torch.int32)
cuda_hist = histogram_module.histogram_cuda(data, 10)
torch_hist = torch.bincount(data, minlength=10)

print("CUDA histogram: ", cuda_hist.tolist())
print("Torch bincount: ", torch_hist.tolist())
print("Match!" if torch.equal(cuda_hist, torch_hist) else "MISMATCH!")
print()
print("Two-level atomics strategy:")
print("  1. atomicAdd in shared memory (fast, ~5 cycles)")
print("  2. Merge into global memory (slow, but only num_bins atomics per block)")

---
## 7. Fused LayerNorm — Production-Quality Kernel

Same operation as the Triton tutorial: fuse mean → variance → normalize → scale+shift.
This shows a realistic CUDA kernel with multiple reduction passes.

In [ ]:
layernorm_cuda_src = r'''
#include <torch/extension.h>
#include <math.h>

// One block per row. Threads cooperate to compute mean and variance.
__global__ void layernorm_kernel(
    const float* input, float* output,
    const float* weight, const float* bias,
    int N, float eps
) {
    extern __shared__ float sdata[];

    int row = blockIdx.x;
    int tid = threadIdx.x;
    int row_offset = row * N;

    // --- Pass 1: Compute mean ---
    float thread_sum = 0.0f;
    for (int col = tid; col < N; col += blockDim.x)
        thread_sum += input[row_offset + col];

    sdata[tid] = thread_sum;
    __syncthreads();
    for (int s = blockDim.x / 2; s > 0; s >>= 1) {
        if (tid < s) sdata[tid] += sdata[tid + s];
        __syncthreads();
    }
    float mean = sdata[0] / N;
    __syncthreads();

    // --- Pass 2: Compute variance ---
    float thread_var = 0.0f;
    for (int col = tid; col < N; col += blockDim.x) {
        float diff = input[row_offset + col] - mean;
        thread_var += diff * diff;
    }

    sdata[tid] = thread_var;
    __syncthreads();
    for (int s = blockDim.x / 2; s > 0; s >>= 1) {
        if (tid < s) sdata[tid] += sdata[tid + s];
        __syncthreads();
    }
    float rstd = rsqrtf(sdata[0] / N + eps);
    __syncthreads();

    // --- Pass 3: Normalize and apply affine transform ---
    for (int col = tid; col < N; col += blockDim.x) {
        float x_hat = (input[row_offset + col] - mean) * rstd;
        output[row_offset + col] = x_hat * weight[col] + bias[col];
    }
}

torch::Tensor layernorm_cuda(
    torch::Tensor input, torch::Tensor weight, torch::Tensor bias, float eps
) {
    TORCH_CHECK(input.dim() == 2, "Need 2D input");
    int M = input.size(0), N = input.size(1);
    auto output = torch::empty_like(input);

    int threads = std::min(1024, N);
    threads = 1;
    while (threads < N && threads < 1024) threads <<= 1;

    layernorm_kernel<<<M, threads, threads * sizeof(float)>>>(
        input.data_ptr<float>(), output.data_ptr<float>(),
        weight.data_ptr<float>(), bias.data_ptr<float>(),
        N, eps
    );
    return output;
}
'''

layernorm_module = load_inline(
    name='layernorm_module',
    cpp_sources='torch::Tensor layernorm_cuda(torch::Tensor input, torch::Tensor weight, torch::Tensor bias, float eps);',
    cuda_sources=layernorm_cuda_src,
    functions=['layernorm_cuda'],
    verbose=False,
)

M, N = 256, 768
x = torch.randn(M, N, device='cuda')
w = torch.ones(N, device='cuda')
b = torch.zeros(N, device='cuda')

cuda_out = layernorm_module.layernorm_cuda(x, w, b, 1e-5)
torch_out = torch.nn.functional.layer_norm(x, [N], w, b)

print(f"Shape: ({M}, {N}) — typical transformer hidden dim")
print(f"Max diff: {(cuda_out - torch_out).abs().max():.2e}")
print("Correct!" if torch.allclose(cuda_out, torch_out, atol=1e-4) else "MISMATCH!")

---
## 8. Benchmarking: CUDA vs PyTorch

In [ ]:
import time

def benchmark(fn, *args, warmup=10, rep=100):
    """Simple GPU benchmark: returns time in ms."""
    for _ in range(warmup):
        fn(*args)
    torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(rep):
        fn(*args)
    torch.cuda.synchronize()
    elapsed_ms = (time.perf_counter() - start) / rep * 1000
    return elapsed_ms

print("=== Vector Addition ===")
size = 10_000_000
x = torch.rand(size, device='cuda')
y = torch.rand(size, device='cuda')

t_cuda = benchmark(add_module.add_cuda, x, y)
t_torch = benchmark(torch.add, x, y)
gbps = lambda ms: 3 * size * 4 * 1e-9 / (ms * 1e-3)
print(f"  CUDA:    {t_cuda:.3f} ms  ({gbps(t_cuda):.1f} GB/s)")
print(f"  PyTorch: {t_torch:.3f} ms  ({gbps(t_torch):.1f} GB/s)")
print()

print("=== Softmax (128 × 4096) ===")
x = torch.randn(128, 4096, device='cuda')
t_cuda = benchmark(softmax_module.softmax_cuda, x)
t_torch = benchmark(lambda x: torch.softmax(x, dim=1), x)
print(f"  CUDA:    {t_cuda:.3f} ms")
print(f"  PyTorch: {t_torch:.3f} ms")
print()

print("=== MatMul (1024 × 1024) ===")
a = torch.randn(1024, 1024, device='cuda')
b = torch.randn(1024, 1024, device='cuda')
t_cuda = benchmark(matmul_module.matmul_cuda, a, b)
t_torch = benchmark(torch.mm, a, b)
print(f"  CUDA (our kernel):  {t_cuda:.3f} ms")
print(f"  PyTorch (cuBLAS):   {t_torch:.3f} ms")
print(f"  (cuBLAS uses advanced optimizations: tensor cores, register tiling, etc.)")

---
## 9. Essential CUDA C Reference

### Qualifiers
| Qualifier | Runs on | Called from |
|---|---|---|
| `__global__` | GPU | CPU (or GPU with dynamic parallelism) |
| `__device__` | GPU | GPU only |
| `__host__` | CPU | CPU only |
| `__host__ __device__` | Both | Both |

### Built-in Variables
| Variable | Description |
|---|---|
| `threadIdx.x/y/z` | Thread index within block |
| `blockIdx.x/y/z` | Block index within grid |
| `blockDim.x/y/z` | Threads per block |
| `gridDim.x/y/z` | Blocks per grid |
| `warpSize` | Warp size (always 32) |

### Synchronization
| Function | Description |
|---|---|
| `__syncthreads()` | Block-level barrier |
| `__syncwarp(mask)` | Warp-level barrier |
| `atomicAdd(addr, val)` | Thread-safe addition |
| `atomicMax(addr, val)` | Thread-safe max |
| `atomicCAS(addr, compare, val)` | Compare-and-swap |

### Math Functions
| Function | Description |
|---|---|
| `expf(x)` | Exponential (float) |
| `logf(x)` | Natural log |
| `sqrtf(x)` | Square root |
| `rsqrtf(x)` | Reciprocal square root (1/sqrt) |
| `fmaxf(x, y)` | Max of two floats |
| `fminf(x, y)` | Min of two floats |
| `fabsf(x)` | Absolute value |
| `__expf(x)` | Fast (less precise) exponential |

### Memory
| Declaration | Type | Scope | Lifetime |
|---|---|---|---|
| `float x;` | Register | Thread | Thread |
| `__shared__ float s[N];` | Shared | Block | Block |
| `__constant__ float c[N];` | Constant | Grid | App |
| `float* ptr` (param) | Global | Grid | Explicit |

---
## 10. Common Patterns & Tips

### Pattern 1: Global Thread Index (1D)
```cpp
int idx = blockIdx.x * blockDim.x + threadIdx.x;
if (idx < n) { /* process element */ }
```

### Pattern 2: Grid-Stride Loop (when elements > threads)
```cpp
// Each thread handles multiple elements, strided by total thread count
int stride = blockDim.x * gridDim.x;
for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
    output[i] = process(input[i]);
}
```

### Pattern 3: 2D Thread Index
```cpp
int row = blockIdx.y * blockDim.y + threadIdx.y;
int col = blockIdx.x * blockDim.x + threadIdx.x;
if (row < M && col < N) {
    C[row * N + col] = compute(row, col);
}
```

### Pattern 4: Shared Memory Reduction
```cpp
__shared__ float sdata[256];
sdata[tid] = my_value;
__syncthreads();
for (int s = blockDim.x / 2; s > 0; s >>= 1) {
    if (tid < s) sdata[tid] += sdata[tid + s];
    __syncthreads();
}
float result = sdata[0];  // only valid in thread 0
```

### Pattern 5: Warp-Level Primitives (Modern CUDA)
```cpp
// Warp shuffle: no shared memory needed for intra-warp communication
float val = my_value;
for (int offset = 16; offset > 0; offset >>= 1)
    val += __shfl_down_sync(0xffffffff, val, offset);
// val in lane 0 now contains the warp-level sum
```

### Performance Tips

1. **Coalesce memory accesses** — adjacent threads should access adjacent memory
   ```
   Good: thread i reads data[i]       (coalesced → one memory transaction)
   Bad:  thread i reads data[i * 100] (strided → many transactions)
   ```

2. **Avoid warp divergence** — threads in a warp should take the same branch
   ```cpp
   // Bad: alternating threads diverge
   if (threadIdx.x % 2 == 0) { path_a(); } else { path_b(); }
   // Better: first half vs second half
   if (threadIdx.x < 16) { path_a(); } else { path_b(); }
   ```

3. **Use enough threads** — need many threads to hide memory latency (occupancy)

4. **Avoid shared memory bank conflicts** — 32 banks, stride-1 access is ideal

5. **Use `__restrict__`** — tells compiler pointers don't alias, enabling optimizations
   ```cpp
   __global__ void kernel(const float* __restrict__ input,
                          float* __restrict__ output, int n)
   ```

---
## 11. Debugging CUDA Kernels

### Strategy 1: Test with tiny inputs
```cpp
// Use 1 block, few threads — easy to reason about
kernel<<<1, 4>>>(data, 4);
```

### Strategy 2: Check for CUDA errors
```cpp
kernel<<<blocks, threads>>>(...);
cudaError_t err = cudaGetLastError();
if (err != cudaSuccess)
    printf("CUDA error: %s\n", cudaGetErrorString(err));
cudaDeviceSynchronize();  // catch async errors
```

### Strategy 3: Use `printf` inside kernels (small scale only!)
```cpp
__global__ void debug_kernel(float* data, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n && idx < 5)  // only print first 5
        printf("Thread %d: data = %f\n", idx, data[idx]);
}
```

### Strategy 4: Write a CPU reference and compare
```python
cuda_result = my_cuda_function(input)
torch_result = torch_reference(input)
print(torch.allclose(cuda_result, torch_result, atol=1e-5))
print((cuda_result - torch_result).abs().max())
```

### Strategy 5: compute-sanitizer (run from terminal)
```bash
compute-sanitizer --tool memcheck python my_script.py
compute-sanitizer --tool racecheck python my_script.py
```

### Common Bugs Checklist
- [ ] Missing `__syncthreads()` before reading shared memory
- [ ] Off-by-one in boundary checks (`<` vs `<=`)
- [ ] Wrong stride/offset for 2D indexing
- [ ] Race conditions — multiple threads writing same location without atomics
- [ ] Uninitialized shared memory
- [ ] Block size not matching shared memory allocation

---
## 12. Side-by-Side: CUDA vs Triton

Let's directly compare the same operation (vector add) in both languages:

### CUDA
```cpp
__global__ void add_kernel(float* x, float* y, float* out, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n)
        out[idx] = x[idx] + y[idx];
}
// Launch: add_kernel<<<(n+255)/256, 256>>>(x, y, out, n);
```

### Triton
```python
@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    tl.store(out_ptr + offs, tl.load(x_ptr + offs, mask=mask) +
             tl.load(y_ptr + offs, mask=mask), mask=mask)
# Launch: add_kernel[grid](x, y, out, n, BLOCK=1024)
```

### When to use which?

**Use Triton when:**
- You want to fuse memory-bound ops (softmax, layernorm, etc.)
- You want rapid iteration and don't need maximum control
- The operation maps naturally to block/tile processing

**Use CUDA when:**
- You need warp-level primitives (`__shfl_sync`, cooperative groups)
- You need fine-grained shared memory control
- You're targeting non-NVIDIA hardware (via HIP/ROCm)
- You need async memory copies (cp.async, TMA)
- Maximum performance is critical and you can invest the time

---
## 13. Exercises

Try implementing these CUDA kernels to practice:

1. **GELU activation**: `0.5 * x * (1 + tanh(sqrt(2/π) * (x + 0.044715 * x³)))`
   - Hint: Use `tanhf()` and `sqrtf()`

2. **Fused Bias + ReLU**: Load input, add a bias vector (broadcasted), apply ReLU
   - Hint: Input is (M, N), bias is (N,). Use `fmaxf(val, 0.0f)`

3. **Parallel prefix sum (scan)**: Given [1,2,3,4], produce [1,3,6,10]
   - Hint: Use shared memory with a up-sweep/down-sweep pattern

4. **2D convolution**: Convolve an image with a 3×3 filter
   - Hint: Load a tile + halo into shared memory

5. **Transpose**: Efficiently transpose a matrix using shared memory
   - Hint: Avoid bank conflicts by padding shared memory `s[TILE][TILE+1]`

### Resources

- [CUDA C Programming Guide](https://docs.nvidia.com/cuda/cuda-c-programming-guide/) — the definitive reference
- [CUDA by Example (book)](https://developer.nvidia.com/cuda-example) — great for beginners
- [GPU Gems](https://developer.nvidia.com/gpugems/gpugems/contributors) — classic optimization techniques
- [CUTLASS](https://github.com/NVIDIA/cutlass) — production GEMM templates
- [Nsight Compute](https://developer.nvidia.com/nsight-compute) — profiling tool